In [ ]:
from pathlib import Path
import re

import pandas as pd


# ============================================================================
# 16. DOCUMENT-ID HEADER LEAK CHECK (urgent, narrow)
# ============================================================================
# 15's institution audit turned up direct evidence that at least some ICLE
# essay TEXT contains a literal corpus document-ID header
# (e.g. "<ICLE-GE-AUG-0033.2>") rather than pure essay content -- visible in
# the printed snippets, not inferred. That header encodes country code and
# institution code directly, in plain text, before the essay even starts.
# If this is present broadly, it is a data-hygiene bug upstream of the
# self-disclosure (11), topic-bias (12), and institution (15) audits -- not
# a fourth confound alongside them, but a likely explanation underneath all
# three. This notebook answers ONE question as directly as possible: how
# much of the corpus has this, in which columns, and where in the text.
#
# Deliberately minimal: it does not replace 11/12/15, but establishes whether
# those audits need to be re-run after a text-cleaning fix, before further
# diagnostic layers are built on contaminated input.

BASE_DIR = Path("..").resolve()
COMBINED_CSV_PATH = BASE_DIR / "data/processed/combined_nli_masked.csv"

# Matches the pattern seen in the actual snippets: <ICLE-XX-YYY(-Y...)-####.#>
# Kept loose (variable-length institution code, optional extra hyphenated segments)
# since the exact convention across all 2619 ICLE files isn't known yet.
HEADER_PATTERN = re.compile(r"<ICLE-[A-Za-z]{2}-[A-Za-z0-9\-]+-\d+\.\d+>")

if not COMBINED_CSV_PATH.exists():
    print(f"Skipping -- {COMBINED_CSV_PATH.name} not found.")
else:
    df = pd.read_csv(COMBINED_CSV_PATH)
    df["dataset"] = df["dataset"].astype(str).str.upper()
    df["text"] = df["text"].fillna("").astype(str)
    df["masked_text"] = df["masked_text"].fillna("").astype(str) if "masked_text" in df.columns else ""

    for col in ["text", "masked_text"]:
        if col not in df.columns:
            continue
        df[f"{col}_has_header"] = df[col].str.contains(HEADER_PATTERN, regex=True)
        df[f"{col}_header_pos"] = df[col].apply(lambda s: (m.start() if (m := HEADER_PATTERN.search(s)) else -1))

    print("=== Rate of document-ID header presence, by dataset and column ===")
    for col in ["text", "masked_text"]:
        flag_col = f"{col}_has_header"
        if flag_col not in df.columns:
            continue
        summary = df.groupby("dataset")[flag_col].agg(n="size", n_flagged="sum")
        summary["pct_flagged"] = (summary["n_flagged"] / summary["n"] * 100).round(1)
        print(f"\n--- {col} ---")
        print(summary)

    icle_with_header = df[(df["dataset"] == "ICLEV3") & df.get("text_has_header", False)]
    if len(icle_with_header):
        pos_stats = icle_with_header["text_header_pos"].describe()
        print(f"\nHeader position within 'text' (character offset from start), ICLE rows with a header:")
        print(pos_stats)
        print("\nSample matches (first 5):")
        for _, row in icle_with_header.head(5).iterrows():
            m = HEADER_PATTERN.search(row["text"])
            context = row["text"][max(0, m.start()-10):m.end()+40] if m else ""
            print(f"  {row['doc_id']}: ...{context}...")

    # Sanity check: does TOEFL11 show anything similar? (Expected: no, different corpus format.)
    toefl_with_header = df[(df["dataset"] == "TOEFL11") & df.get("text_has_header", False)]
    print(f"\nTOEFL11 rows with a matching header pattern: {len(toefl_with_header)} / "
          f"{len(df[df['dataset']=='TOEFL11'])}")

    print(
        "\nIf ICLE's 'text' column is showing a high header-presence rate: every notebook that trained "
        "or evaluated on full-text ICLE (03/04/05/07/08/09/10/11/12/13/15) was exposed to this. "
        "The fix belongs upstream, in whichever notebook builds combined_nli_masked.csv -- strip the "
        "header there, regenerate the CSV, and re-run downstream from that point. Re-running 15/16 "
        "again on the same uncleaned CSV will not change this result."
    )
